<div style="display: flex; align-items: center; padding: 20px; background-color: #f0f2f6; border-radius: 10px; border: 2px solid #007bff;">
    <img src="../logo.png" style="width: 80px; height: auto; margin-right: 20px;">
    <div style="flex: 1; text-align: left;">
    <h1 style="color: #007bff; margin-bottom: 5px;">GLY 6739.017S26: Computational Seismology</h1>
    <h3 style="color: #666;">Notebook 74:  Local Data, Cloud Data, and Synthetic Data with ObsPy</h3>
    <p style="color: red;"><i>Glenn Thompson | Spring 2026</i></p>
    </div>
</div>



# 1. Local files

We can look up information about many stations in the Federation of Digital Seismic Networks (FDSN) **Station Book**

Information about station II.PFO:
https://www.fdsn.org/station_book/II/PFO/pfo_14.html

You can see that has two seismic sensors:
* An STS-1 broadband seismometer
* An FBA-23 accelerometer

If you click on the **Channel Data** link you will see that it reports channels at several sampling rates:
* 1 Hz
* 20 Hz
* 40 Hz
* 100 Hz (triggered only)

Sensitivities are in Counts/m/s

Note that https://earthquake.usgs.gov/monitoring/operations/stations/II/PFO/ lists other types of sensors. I don't know which is authoritative! But fortunately, we have the actual StationXML file!

Let's load it!


In [ ]:
from obspy import read_inventory
inv = read_inventory("../mess2024/data/station_PFO.xml", format="STATIONXML")
#inv.get_response(st[0].id, st[0].stats.starttime).plot(0.01);

for net in inv:
    for sta in net:
        for cha in sta:
            print(cha)
            cha.response.plot(0.01);

So both the FDSN and USGS sites were half right. Sensor 1 is an STS-1, sensor 2 is a Trillium 240!

Let's read a locally-stored MiniSEED file for station II.PFO:

In [ ]:
from obspy import read, read_inventory
st = read("../mess2024/data/waveform_PFO.mseed")
print(st)
st.plot(equal_scale=False);


The waveform shapes are quite similar, but the amplitudes are different by ~3. Why?

Also, the top trace has a 20 Hz sampling rate, the bottom has a 40 Hz sampling rate. Let's resample the bottom one:

In [ ]:
st[1].resample(sampling_rate=20.0)
print(st)

As you see, there are two Traces, and they have different location codes (00 vs 10). These are from the two different sensors. They are essentially at the same latitude/longitude, but might differ in depth. So we should see the same waveforms right?

Mathematically, we can check that by cross-correlating the signals. To do that, we must make sure they have the same sampling rate - and we see from above that II.PFO.00.BHZ is 20 Hz data, while II.PFO.10.BHZ is 40 Hz data. 



Let's cross correlate the signals from these co-located vertical sensors:

In [ ]:
from obspy.signal.cross_correlation import correlate, xcorr_max
import matplotlib.pyplot as plt

def plot_xcorr(st_in):
    st = st_in.copy()
    fs = min([tr.stats.sampling_rate for tr in st])
    st.resample(sampling_rate=fs)
    cc = correlate(st[0], st[1], int(st[0].stats.sampling_rate))

    plt.figure()
    plt.plot(cc)
    plt.xlabel('Shift (samples)')
    plt.ylabel('Cross correlation coefficient')

    shift, value = xcorr_max(cc)
    print(f'max xcorr is {value} at a shift of {shift} samples')

plot_xcorr(st)

The peak correlation value is 99.27% - very high! But wait, that occurs at a shift of 3 samples. 3 samples at 20 Hz (0.05 s sampling interval) is 0.15 s. Even at a (surface) wave speed of 4 km/s, that suggests the sensors are ~600 m apart!

**What is going on here?**

In [ ]:
st.detrend("linear")
st.taper(max_percentage=0.05, type='cosine')
st.filter("bandpass", freqmin=0.01, freqmax=0.1)
st.plot();

Let's try one more thing:

In [ ]:
st.remove_response(inventory=inv)
st.plot();
plot_xcorr(st)

**What changed?**

After removing the instrument response:
* amplitudes are the same
* shift is now 0 samples
* peak correlation coefficient is now 99.75%

# response removal can vary

**water_level** and **pre_filt** are two stabilization tools used during instrument response removal, but they play different roles. 

**water_level** is a regularization parameter applied in the frequency domain during deconvolution. When removing the instrument response, we divide the data spectrum by the response spectrum. At frequencies where the response magnitude becomes very small (for example, outside the flat passband), this division can amplify noise dramatically. The water level sets a minimum floor on the response amplitude—expressed in decibels relative to the peak response—so that the inverse filter cannot exceed a specified amplification factor. It keeps the math stable and prevents catastrophic noise blow-up, but it does not define a physical bandwidth.

**pre_filt**, by contrast, defines the frequency range over which we trust the inversion. It is a four-corner cosine taper (f1, f2, f3, f4) applied in the frequency domain before deconvolution. Between f2 and f3 the inversion is trusted; below f1 and above f4 energy is suppressed; and the regions f1–f2 and f3–f4 are smooth tapers. This is not a “filter” in the usual time-domain sense, but rather a bandwidth constraint that prevents us from trying to recover signal where the instrument response or noise level makes the result unreliable. In short: water level stabilizes the inversion; pre_filt defines the usable frequency band.

We can also use the **output** parameter to select whether to correct to VELocity, ACCeleration, or DISPlacement seismograms. When you integrate (e.g. VEL -> DISP), you blow up low frequencies relative to higher frequencies and add a constant. When you differentiate (e.g. VEL -> ACC) you diminish low frequencies relative to higher frequencies.

In [ ]:
st = read("../mess2024/data/waveform_PFO.mseed")
st.remove_response(inventory=inv, water_level=60, pre_filt=(0.001, 0.002, 8, 10), output="DISP")
st.resample(sampling_rate=20.0)
st.plot();
plot_xcorr(st)

st = read("../mess2024/data/waveform_PFO.mseed")
st.remove_response(inventory=inv, water_level=60, output="ACC")
st.resample(sampling_rate=20.0)
st.plot();
plot_xcorr(st)

Finally, this is how we read events from a local QuakeML file into an ObsPy Catalog object:

In [ ]:
from obspy import read_events

catalog = read_events("../mess2024/data/event_tohoku_with_big_aftershocks.xml")
print(catalog)

# 3. Make our own data

Waveforms:

In [ ]:
from obspy import Stream, Trace, UTCDateTime

x = np.random.randint(-100, 100, 500)
tr = Trace(data=x)
tr.stats.station = "XYZ"
tr.stats.starttime = UTCDateTime()

tr2 = Trace(data=np.random.randint(-300, 100, 1000))
tr2.stats.starttime = UTCDateTime()
tr2.stats.sampling_rate = 10.0
st = Stream([tr, tr2])

print(st)
st.plot();



Events 

In [ ]:
from obspy import UTCDateTime
from obspy.core.event import Catalog, Event, Origin, Magnitude
from obspy.geodetics import FlinnEngdahl

cat = Catalog()
cat.description = "Just a fictitious toy example catalog built from scratch"

e = Event()
e.event_type = "not existing"

o = Origin()
o.time = UTCDateTime(2014, 2, 23, 18, 0, 0)
o.latitude = 47.6
o.longitude = 12.0
o.depth = 10000
o.depth_type = "operator assigned"
o.evaluation_mode = "manual"
o.evaluation_status = "preliminary"
o.region = FlinnEngdahl().get_region(o.longitude, o.latitude)

m = Magnitude()
m.mag = 7.2
m.magnitude_type = "Mw"

m2 = Magnitude()
m2.mag = 7.4
m2.magnitude_type = "Ms"

# also included could be: custom picks, amplitude measurements, station magnitudes,
# focal mechanisms, moment tensors, ...

# make associations, put everything together
cat.append(e)
e.origins = [o]
e.magnitudes = [m, m2]
m.origin_id = o.resource_id
m2.origin_id = o.resource_id

print(cat)
cat.write("/tmp/my_custom_events.xml", format="QUAKEML")
!cat /tmp/my_custom_events.xml

### FDSN

ObsPy has clients to directly fetch data via...

- FDSN webservices (IRIS, Geofon/GFZ, USGS, NCEDC, SeisComp3 instances, ...)
- ArcLink (EIDA, ...)
- Earthworm
- SeedLink (near-realtime servers)
- NERIES/NERA/seismicportal.eu
- NEIC
- SeisHub (local seismological database)

This introduction shows how to use the FDSN webservice client. The FDSN webservice definition is by now the default web service implemented by many data centers world wide. Clients for other protocols work similar to the FDSN client.

#### Waveform Data

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")
t = UTCDateTime("2011-03-11T05:46:23")  # Tohoku
st = client.get_waveforms("II", "PFO", "*", "LHZ",
                          t + 10 * 60, t + 30 * 60)
print(st)
st.plot();

In [ ]:
import obspy
from obspy.clients.fdsn import Client

c_event = Client("USGS")

# Event time.
event_time = obspy.UTCDateTime("2011-03-11T05:46:23.2")

# Get the event information. The temporal and magnitude constraints make it unique
cat = c_event.get_events(starttime=event_time - 10, endtime=event_time + 10,
                         minmagnitude=9)
print(cat)

c = Client("IRIS")
# Download station information at the response level!
inv = c.get_stations(network="II", station="BFO", location="*", channel="BH?",
                     starttime=event_time - 60, endtime=event_time + 3600,
                     level="response")
print(inv)

# Download 3 component waveforms.
st = c.get_waveforms(network="II", station="BFO", location="*",
                     channel="BH?", starttime=event_time - 60,
                     endtime=event_time + 3600)
print(st)